In [ ]:
import os
from pathlib import Path
import spiceypy as spice
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from mpl_toolkits.mplot3d import Axes3D
import re

In [ ]:
import spiceypy as spice
print(spice.tkvrsn('TOOLKIT'))

In [ ]:
# list the files under spice/extras/mk
print("Files under spice/extras/mk:")
for file in (base / "extras" / "mk").iterdir():
    print(file.name)

In [ ]:
base

In [ ]:
import os
from pathlib import Path
import spiceypy as spice

spice.kclear()

spice_root = Path("/Users/chengzhimao/Documents/Code/saturn2025/data/spice")
mk = spice_root / "extras" / "mk" / "cas_2004_v27.tm"

old_cwd = Path.cwd()
os.chdir(spice_root)

try:
    spice.furnsh(str(mk))
finally:
    os.chdir(old_cwd)

print("Loaded kernels:", spice.ktotal("ALL"))

In [ ]:
saturn_radii = spice.bodvrd("SATURN", "RADII", 3)[1]
RS_km = saturn_radii[0]

In [ ]:
RS_km
# Saturn radius in km: 60268.0

In [ ]:
import os
from pathlib import Path
import spiceypy as spice
import numpy as np

spice.kclear()

spice_root = Path("/Users/chengzhimao/Documents/Code/saturn2025/data/spice")
mk = spice_root / "extras" / "mk" / "cas_2004_v27.tm"

# Stay in spice_root while using the kernels
os.chdir(spice_root)

spice.furnsh(str(mk))

print("Current folder:", Path.cwd())
print("Loaded kernels:", spice.ktotal("ALL"))

utc = "2004-07-01T00:00:00"
et = spice.utc2et(utc)

state, light_time = spice.spkezr(
    "CASSINI",
    et,
    "J2000",
    "NONE",
    "SATURN"
)

position_km = state[:3]
velocity_km_s = state[3:]
speed_km_s = np.linalg.norm(velocity_km_s)

print("Position relative to Saturn [km]:", position_km)
print("Velocity relative to Saturn [km/s]:", velocity_km_s)
print("Speed relative to Saturn [km/s]:", speed_km_s)

In [ ]:
import pandas as pd
import numpy as np

start_utc = "2004-01-01T00:00:00"
stop_utc  = "2004-01-03T00:00:00"   # test with 2 days first

start_et = spice.utc2et(start_utc)
stop_et = spice.utc2et(stop_utc)

step_seconds = 3600  # 1 hour
ets = np.arange(start_et, stop_et + step_seconds, step_seconds)

rows = []

for et_i in ets:
    utc_i = spice.et2utc(et_i, "ISOC", 3)

    state, lt = spice.spkezr(
        "CASSINI",
        et_i,
        "CASSINI_KRTP",
        "NONE",
        "SATURN"
    )

    rtp_pos = state[:3]
    rtp_vel = state[3:]

    rows.append({
        "utc": utc_i,
        "R_km": rtp_pos[0],
        "Theta_km": rtp_pos[1],
        "Phi_km": rtp_pos[2],
        "vR_km_s": rtp_vel[0],
        "vTheta_km_s": rtp_vel[1],
        "vPhi_km_s": rtp_vel[2],
        "speed_km_s": np.linalg.norm(rtp_vel),
    })

df = pd.DataFrame(rows)
df.head()

In [ ]:
state

# Load in spice

In [ ]:
def latest_tm_for_year(year):
    files = list(mk_dir.glob(f"cas_{year}_v*.tm"))

    if not files:
        raise FileNotFoundError(f"No meta-kernel found for year {year}")

    def version_number(path):
        m = re.search(r"_v(\d+)\.tm$", path.name)
        return int(m.group(1))

    return max(files, key=version_number)


def load_cassini_year(year):
    spice.kclear()

    mk = latest_tm_for_year(year)
    spice.furnsh(str(mk))

    print(f"Loaded {mk.name}; kernels loaded = {spice.ktotal('ALL')}")

    return mk

## KRTP

In [ ]:
# Paths
spice_root = Path("/Users/chengzhimao/Documents/Code/saturn2025/data/spice")
mk_dir = spice_root / "extras" / "mk"

out_file = Path("/Users/chengzhimao/Documents/Code/saturn2025/cassini_krtp_2004_2017_60s.pkl")

# Important: stay in spice_root because the Cassini .tm files use relative paths like ./data/spk/...
os.chdir(spice_root)

print("Working directory:", Path.cwd())

In [ ]:
def generate_one_year_krtp(year, step_seconds=60):
    load_cassini_year(year)

    start_utc = f"{year}-01-01T00:00:00"

    if year < 2017:
        stop_utc = f"{year + 1}-01-01T00:00:00"
    else:
        # Conservative stop before final Cassini coverage problem
        stop_utc = "2017-09-15T10:40:00"

    print(f"Generating {year}: {start_utc} to {stop_utc}")

    utc_times = pd.date_range(
        start=start_utc,
        end=stop_utc,
        freq=f"{step_seconds}s",
        inclusive="left",
        tz="UTC"
    )

    utc_strings = utc_times.strftime("%Y-%m-%dT%H:%M:%S").to_numpy()
    ets = np.array([spice.utc2et(t) for t in utc_strings], dtype=np.float64)

    states, light_times = spice.spkezr(
        "CASSINI",
        ets,
        "CASSINI_KRTP",
        "NONE",
        "SATURN"
    )

    states = np.asarray(states, dtype=np.float64)

    df = pd.DataFrame({
        "utc": utc_times,
        "et": ets,
        "R_km": states[:, 0],
        "Theta_km": states[:, 1],
        "Phi_km": states[:, 2],
        "vR_km_s": states[:, 3],
        "vTheta_km_s": states[:, 4],
        "vPhi_km_s": states[:, 5],
    })

    df["speed_km_s"] = np.linalg.norm(states[:, 3:6], axis=1)

    return df

In [ ]:
step_seconds = 60

dfs = []

for year in range(2004, 2018):
    df_year = generate_one_year_krtp(year, step_seconds=step_seconds)
    print(f"{year}: rows = {len(df_year):,}")
    dfs.append(df_year)

df_all = pd.concat(dfs, ignore_index=True)

df_all.attrs["target"] = "CASSINI"
df_all.attrs["observer"] = "SATURN"
df_all.attrs["frame"] = "CASSINI_KRTP"
df_all.attrs["abcorr"] = "NONE"
df_all.attrs["step_seconds"] = step_seconds
df_all.attrs["start_utc"] = "2004-01-01T00:00:00"
df_all.attrs["stop_utc"] = "2017-09-16T00:00:00"

print("Total rows:", len(df_all))
print(df_all.head())
print(df_all.tail())

In [ ]:
df_all.to_pickle(out_file, protocol=5)

size_gb = out_file.stat().st_size / 1024**3
print("Saved:", out_file)
print(f"File size: {size_gb:.3f} GB")

## J2000
centered on Saturn
x-axis points towards the vernal equinox
z-axis points towards the north pole of Earth's rotation axis

In [ ]:
# Main paths
spice_root = Path("/Users/chengzhimao/Documents/Code/saturn2025/data/spice")
mk_dir = spice_root / "extras" / "mk"

out_file = Path(
    "/Users/chengzhimao/Documents/Code/saturn2025/"
    "cassini_j2000_saturn_centered_2004_2017_60s.pkl"
)

# Important: stay in spice_root because Cassini .tm files use relative paths like ./data/spk/...
os.chdir(spice_root)

print("Working directory:", Path.cwd())

In [ ]:
def generate_one_year_j2000(year, step_seconds=60):
    load_cassini_year(year)

    start_utc = f"{year}-01-01T00:00:00"

    if year < 2017:
        stop_utc = f"{year + 1}-01-01T00:00:00"
    else:
        # Conservative stop before final Cassini SPK coverage issue
        stop_utc = "2017-09-15T10:40:00"

    print(f"Generating {year}: {start_utc} to {stop_utc}")

    utc_times = pd.date_range(
        start=start_utc,
        end=stop_utc,
        freq=f"{step_seconds}s",
        inclusive="left",
        tz="UTC"
    )

    utc_strings = utc_times.strftime("%Y-%m-%dT%H:%M:%S").to_numpy()

    ets = np.array(
        [spice.utc2et(t) for t in utc_strings],
        dtype=np.float64
    )

    states, light_times = spice.spkezr(
        "CASSINI",
        ets,
        "J2000",
        "NONE",
        "SATURN"
    )

    states = np.asarray(states, dtype=np.float64)

    # Saturn radius, useful for plotting in units of R_S
    saturn_radii = spice.bodvrd("SATURN", "RADII", 3)[1]
    RS_km = saturn_radii[0]

    df = pd.DataFrame({
        "utc": utc_times,
        "et": ets,

        # Saturn-centered J2000 position [km]
        "x_km": states[:, 0],
        "y_km": states[:, 1],
        "z_km": states[:, 2],

        # Saturn-centered J2000 velocity [km/s]
        "vx_km_s": states[:, 3],
        "vy_km_s": states[:, 4],
        "vz_km_s": states[:, 5],
    })

    df["r_km"] = np.linalg.norm(states[:, 0:3], axis=1)
    df["speed_km_s"] = np.linalg.norm(states[:, 3:6], axis=1)

    # Same position in Saturn radii
    df["x_Rs"] = df["x_km"] / RS_km
    df["y_Rs"] = df["y_km"] / RS_km
    df["z_Rs"] = df["z_km"] / RS_km
    df["r_Rs"] = df["r_km"] / RS_km

    return df

In [ ]:
step_seconds = 60

dfs = []

for year in range(2004, 2018):
    df_year = generate_one_year_j2000(year, step_seconds=step_seconds)
    print(f"{year}: rows = {len(df_year):,}")
    dfs.append(df_year)

df_all = pd.concat(dfs, ignore_index=True)

df_all.attrs["target"] = "CASSINI"
df_all.attrs["observer"] = "SATURN"
df_all.attrs["frame"] = "J2000"
df_all.attrs["abcorr"] = "NONE"
df_all.attrs["step_seconds"] = step_seconds
df_all.attrs["start_utc"] = "2004-01-01T00:00:00"
df_all.attrs["stop_utc"] = "2017-09-15T10:40:00"
df_all.attrs["description"] = (
    "Cassini position and velocity relative to Saturn, expressed in J2000 Cartesian coordinates."
)

df_all.to_pickle(out_file, protocol=5)

print("Saved:", out_file)
print("Rows:", len(df_all))
print(f"File size: {out_file.stat().st_size / 1024**3:.3f} GB")

## KSM
Kronocentric Solar Magnetospheric
x-axis points from Saturn to the Sun
z-axis is the projection of Saturn's north pole onto the plane perpendicular to the x-axis?

In [ ]:
spice_root = Path("/Users/chengzhimao/Documents/Code/saturn2025/data/spice")
mk_dir = spice_root / "extras" / "mk"

out_file = Path(
    "/Users/chengzhimao/Documents/Code/saturn2025/"
    "cassini_ksm_2004_2017_60s.pkl"
)

# Important because Cassini .tm files use relative paths like ./data/spk/...
os.chdir(spice_root)

print("Working directory:", Path.cwd())

In [ ]:
def generate_one_year_ksm(year, step_seconds=60):
    load_cassini_year(year)

    start_utc = f"{year}-01-01T00:00:00"

    if year < 2017:
        stop_utc = f"{year + 1}-01-01T00:00:00"
    else:
        # Conservative stop before final 2017 SPK coverage issue
        stop_utc = "2017-09-15T10:40:00"

    print(f"Generating {year}: {start_utc} to {stop_utc}")

    utc_times = pd.date_range(
        start=start_utc,
        end=stop_utc,
        freq=f"{step_seconds}s",
        inclusive="left",
        tz="UTC"
    )

    utc_strings = utc_times.strftime("%Y-%m-%dT%H:%M:%S").to_numpy()

    ets = np.array(
        [spice.utc2et(t) for t in utc_strings],
        dtype=np.float64
    )

    # Cassini state relative to Saturn, expressed in CASSINI_KSM
    states_ksm, light_times = spice.spkezr(
        "CASSINI",
        ets,
        "CASSINI_KSM",
        "NONE",
        "SATURN"
    )

    states_ksm = np.asarray(states_ksm, dtype=np.float64)

    # Saturn radius
    RS_km = spice.bodvrd("SATURN", "RADII", 3)[1][0]

    x_km = states_ksm[:, 0]
    y_km = states_ksm[:, 1]
    z_km = states_ksm[:, 2]

    vx_km_s = states_ksm[:, 3]
    vy_km_s = states_ksm[:, 4]
    vz_km_s = states_ksm[:, 5]

    x_Rs = x_km / RS_km
    y_Rs = y_km / RS_km
    z_Rs = z_km / RS_km

    r_Rs = np.sqrt(x_Rs**2 + y_Rs**2 + z_Rs**2)

    # Approximate magnetic/local latitude from KSM z
    lat_rad = np.arcsin(z_Rs / r_Rs)
    lat_deg = np.degrees(lat_rad)

    # Dipole-style L approximation:
    # r = L cos^2(lambda), so L = r / cos^2(lambda)
    L_shell = r_Rs / (np.cos(lat_rad)**2)

    # KSM MLT:
    # +X = noon = 12 hr
    # +Y = dusk = 18 hr
    # -X = midnight = 00 hr
    # -Y = dawn = 06 hr
    angle_deg = np.degrees(np.arctan2(y_Rs, x_Rs))
    MLT_hr = (12.0 + angle_deg / 15.0) % 24.0

    speed_km_s = np.sqrt(vx_km_s**2 + vy_km_s**2 + vz_km_s**2)

    df = pd.DataFrame({
        "utc": utc_times,
        "et": ets,

        "x_ksm_km": x_km,
        "y_ksm_km": y_km,
        "z_ksm_km": z_km,

        "x_ksm_Rs": x_Rs,
        "y_ksm_Rs": y_Rs,
        "z_ksm_Rs": z_Rs,

        "r_Rs": r_Rs,
        "L_approx": L_shell,
        "MLT_hr": MLT_hr,
        "MLAT_approx_deg": lat_deg,

        "vx_ksm_km_s": vx_km_s,
        "vy_ksm_km_s": vy_km_s,
        "vz_ksm_km_s": vz_km_s,
        "speed_km_s": speed_km_s,
    })

    return df

In [ ]:
step_seconds = 60

dfs = []

for year in range(2004, 2018):
    print(f"\nProcessing {year}")

    df_year = generate_one_year_ksm(
        year,
        step_seconds=step_seconds
    )

    print(f"{year}: rows = {len(df_year):,}")

    dfs.append(df_year)

df_all = pd.concat(dfs, ignore_index=True)

df_all.attrs["target"] = "CASSINI"
df_all.attrs["observer"] = "SATURN"
df_all.attrs["frame"] = "CASSINI_KSM"
df_all.attrs["abcorr"] = "NONE"
df_all.attrs["step_seconds"] = step_seconds
df_all.attrs["start_utc"] = "2004-01-01T00:00:00"
df_all.attrs["stop_utc"] = "2017-09-15T10:40:00"
df_all.attrs["MLT_convention"] = (
    "+X KSM = 12 MLT/noon, +Y KSM = 18 MLT/dusk, "
    "-X KSM = 00 MLT/midnight, -Y KSM = 06 MLT/dawn"
)
df_all.attrs["L_note"] = "L_approx = r_Rs / cos(MLAT)^2, using KSM z as approximate latitude."

df_all.to_pickle(out_file, protocol=5)

print("Saved:", out_file)
print("Rows:", len(df_all))
print(f"File size: {out_file.stat().st_size / 1024**3:.3f} GB")

## ENIS 
Kriegel et al. 2011 Influence of negatively charged plume

Origin = Enceladus

+X_ENIS = ideal corotation direction u0

+Y_ENIS = Enceladus → Saturn direction
          Saturn-facing direction

+Z_ENIS = northward
          completes right-handed system

Right-handed:
    +X × +Y = +Z

In [ ]:
spice_root = Path("/Users/chengzhimao/Documents/Code/saturn2025/data/spice")
mk_dir = spice_root / "extras" / "mk"

out_file = Path(
    "/Users/chengzhimao/Documents/Code/saturn2025/data/"
    "spice-location-ready/"
    "cassini_enis_2004_2017_60s.pkl"
)

# Important because Cassini .tm files use relative paths like ./data/spk/...
os.chdir(spice_root)

print("Working directory:", Path.cwd())

In [ ]:
out_file = Path(
    "/Users/chengzhimao/Documents/Code/saturn2025/data/"
    "spice-location-ready/"
    "cassini_enis_2004_2017_60s.pkl"
)


out_file

In [ ]:
def unit(v):
    v = np.asarray(v, dtype=np.float64)
    n = np.linalg.norm(v)

    if n == 0:
        raise ValueError("Cannot normalize zero vector.")

    return v / n

In [ ]:
def enis_axes_at_et(et):
    """
    Enceladus Interaction System, ENIS.

    Definition used:

        Origin = Enceladus

        +X = ideal corotation direction u0

        +Y = Enceladus -> Saturn direction
             Saturn-facing direction

        +Z = +X cross +Y
             northward direction

    Returns
    -------
    x_hat, y_hat, z_hat : ndarray
        Unit vectors of ENIS axes expressed in J2000 components.
    """

    # Saturn -> Enceladus, expressed in J2000
    state_enc_from_sat, _ = spice.spkezr(
        "ENCELADUS",
        et,
        "J2000",
        "NONE",
        "SATURN"
    )

    r_sat_to_enc = np.asarray(state_enc_from_sat[:3], dtype=np.float64)

    # +Y_ENIS = Enceladus -> Saturn
    # Since r_sat_to_enc is Saturn -> Enceladus,
    # Enceladus -> Saturn is the negative.
    y_hat = unit(-r_sat_to_enc)

    # Saturn north spin axis, expressed in J2000
    rot_iau_saturn_to_j2000 = spice.pxform(
        "IAU_SATURN",
        "J2000",
        et
    )

    saturn_north = rot_iau_saturn_to_j2000 @ np.array([0.0, 0.0, 1.0])
    saturn_north = unit(saturn_north)

    # +X_ENIS = ideal corotation direction u0
    #
    # Ideal corotation direction is Omega x r,
    # where Omega is Saturn north spin axis,
    # and r is Saturn -> Enceladus.
    x_hat = np.cross(saturn_north, r_sat_to_enc)
    x_hat = unit(x_hat)

    # +Z_ENIS completes right-handed system:
    # X cross Y = Z
    z_hat = np.cross(x_hat, y_hat)
    z_hat = unit(z_hat)

    # Optional: re-orthogonalize Y to avoid tiny numerical drift
    # Keep X and Z fixed, recompute Y so the frame is exactly right-handed.
    y_hat = np.cross(z_hat, x_hat)
    y_hat = unit(y_hat)

    return x_hat, y_hat, z_hat

In [ ]:
def generate_one_year_enis(year, step_seconds=60):
    load_cassini_year(year)

    start_utc = f"{year}-01-01T00:00:00"

    if year < 2017:
        stop_utc = f"{year + 1}-01-01T00:00:00"
    else:
        # Conservative stop before final 2017 SPK coverage issue
        stop_utc = "2017-09-15T10:40:00"

    print(f"Generating {year}: {start_utc} to {stop_utc}")

    utc_times = pd.date_range(
        start=start_utc,
        end=stop_utc,
        freq=f"{step_seconds}s",
        inclusive="left",
        tz="UTC"
    )

    utc_strings = utc_times.strftime("%Y-%m-%dT%H:%M:%S").to_numpy()

    ets = np.array(
        [spice.utc2et(t) for t in utc_strings],
        dtype=np.float64
    )

    # Enceladus radius
    RE_km = np.mean(spice.bodvrd("ENCELADUS", "RADII", 3)[1])

    n = len(ets)

    x_km = np.empty(n, dtype=np.float64)
    y_km = np.empty(n, dtype=np.float64)
    z_km = np.empty(n, dtype=np.float64)

    vx_km_s = np.empty(n, dtype=np.float64)
    vy_km_s = np.empty(n, dtype=np.float64)
    vz_km_s = np.empty(n, dtype=np.float64)

    saturn_x_km = np.empty(n, dtype=np.float64)
    saturn_y_km = np.empty(n, dtype=np.float64)
    saturn_z_km = np.empty(n, dtype=np.float64)

    sun_x_km = np.empty(n, dtype=np.float64)
    sun_y_km = np.empty(n, dtype=np.float64)
    sun_z_km = np.empty(n, dtype=np.float64)

    for i, et in enumerate(ets):
        x_hat, y_hat, z_hat = enis_axes_at_et(et)

        # Enceladus -> Cassini in J2000
        state_cas, _ = spice.spkezr(
            "CASSINI",
            et,
            "J2000",
            "NONE",
            "ENCELADUS"
        )

        r_cas = np.asarray(state_cas[:3], dtype=np.float64)
        v_cas = np.asarray(state_cas[3:6], dtype=np.float64)

        x_km[i] = np.dot(r_cas, x_hat)
        y_km[i] = np.dot(r_cas, y_hat)
        z_km[i] = np.dot(r_cas, z_hat)

        vx_km_s[i] = np.dot(v_cas, x_hat)
        vy_km_s[i] = np.dot(v_cas, y_hat)
        vz_km_s[i] = np.dot(v_cas, z_hat)

        # Enceladus -> Saturn in J2000
        state_sat, _ = spice.spkezr(
            "SATURN",
            et,
            "J2000",
            "NONE",
            "ENCELADUS"
        )

        r_sat = np.asarray(state_sat[:3], dtype=np.float64)

        saturn_x_km[i] = np.dot(r_sat, x_hat)
        saturn_y_km[i] = np.dot(r_sat, y_hat)
        saturn_z_km[i] = np.dot(r_sat, z_hat)

        # Enceladus -> Sun in J2000
        state_sun, _ = spice.spkezr(
            "SUN",
            et,
            "J2000",
            "NONE",
            "ENCELADUS"
        )

        r_sun = np.asarray(state_sun[:3], dtype=np.float64)

        sun_x_km[i] = np.dot(r_sun, x_hat)
        sun_y_km[i] = np.dot(r_sun, y_hat)
        sun_z_km[i] = np.dot(r_sun, z_hat)

    x_RE = x_km / RE_km
    y_RE = y_km / RE_km
    z_RE = z_km / RE_km

    distance_km = np.sqrt(
        x_km**2 +
        y_km**2 +
        z_km**2
    )

    distance_RE = distance_km / RE_km
    altitude_km = distance_km - RE_km

    speed_km_s = np.sqrt(
        vx_km_s**2 +
        vy_km_s**2 +
        vz_km_s**2
    )

    # Local angle in ENIS X-Y plane
    #
    # Convention here:
    #   0 hr  = +Y = Saturn-facing
    #   6 hr  = +X = corotation direction
    #   12 hr = -Y = Saturn-averted
    #   18 hr = -X
    local_angle_deg = np.degrees(np.arctan2(x_RE, y_RE))
    local_time_hr = (local_angle_deg / 15.0) % 24.0

    df = pd.DataFrame({
        "utc": utc_times,
        "et": ets,

        "x_enis_km": x_km,
        "y_enis_km": y_km,
        "z_enis_km": z_km,

        "x_enis_RE": x_RE,
        "y_enis_RE": y_RE,
        "z_enis_RE": z_RE,

        "distance_km": distance_km,
        "distance_RE": distance_RE,
        "altitude_km": altitude_km,

        "local_time_hr": local_time_hr,

        "vx_enis_km_s": vx_km_s,
        "vy_enis_km_s": vy_km_s,
        "vz_enis_km_s": vz_km_s,
        "speed_km_s": speed_km_s,

        "saturn_x_enis_km": saturn_x_km,
        "saturn_y_enis_km": saturn_y_km,
        "saturn_z_enis_km": saturn_z_km,

        "saturn_x_enis_RE": saturn_x_km / RE_km,
        "saturn_y_enis_RE": saturn_y_km / RE_km,
        "saturn_z_enis_RE": saturn_z_km / RE_km,

        "sun_x_enis_km": sun_x_km,
        "sun_y_enis_km": sun_y_km,
        "sun_z_enis_km": sun_z_km,

        "sun_x_enis_RE": sun_x_km / RE_km,
        "sun_y_enis_RE": sun_y_km / RE_km,
        "sun_z_enis_RE": sun_z_km / RE_km,
    })

    return df

In [ ]:
step_seconds = 60

dfs = []

for year in range(2004, 2018):
    print(f"\nProcessing {year}")

    df_year = generate_one_year_enis(
        year,
        step_seconds=step_seconds
    )

    print(f"{year}: rows = {len(df_year):,}")

    dfs.append(df_year)

df_all = pd.concat(dfs, ignore_index=True)

df_all.attrs["target"] = "CASSINI"
df_all.attrs["observer"] = "ENCELADUS"
df_all.attrs["frame"] = "custom_ENIS_official_definition"
df_all.attrs["abcorr"] = "NONE"
df_all.attrs["step_seconds"] = step_seconds
df_all.attrs["start_utc"] = "2004-01-01T00:00:00"
df_all.attrs["stop_utc"] = "2017-09-15T10:40:00"

df_all.attrs["axis_definition"] = (
    "Origin = Enceladus; "
    "+X = ideal corotation direction u0 = Saturn spin axis cross Saturn->Enceladus; "
    "+Y = Enceladus -> Saturn, Saturn-facing direction; "
    "+Z = +X cross +Y, northward."
)

df_all.attrs["local_time_convention"] = (
    "+Y = 0 hr, Saturn-facing; "
    "+X = 6 hr, corotation direction; "
    "-Y = 12 hr, Saturn-averted; "
    "-X = 18 hr."
)



In [17]:
df_all.to_pickle(out_file, protocol=5)

print("Saved:", out_file)
print("Rows:", len(df_all))
print(f"File size: {out_file.stat().st_size / 1024**3:.3f} GB")

Saved: /Users/chengzhimao/Documents/Code/saturn2025/data/spice-location-ready/cassini_enis_2004_2017_60s.pkl
Rows: 7209280
File size: 1.504 GB


# After SPICE

## KRTP

In [ ]:
with open(Path("/Users/chengzhimao/Documents/Code/saturn2025/cassini_krtp_2004_2017_60s.pkl"), "rb") as f:
    df_loaded = pd.read_pickle(f)

In [ ]:
df_loaded

In [ ]:
print(df_loaded.head())

In [ ]:
# plot the location in R-Theta for the first 1000 points, in polar coordinates (R, Theta), 

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(df_loaded["R_km"][1000:1000_1000], marker='o', linestyle='-', markersize=3)

In [ ]:
# plot a distribution of r from 2015-2017 Sep 1

start_utc = "2015-09-01T00:00:00"
stop_utc = "2017-09-01T00:00:00"

mask = (df_loaded["utc"] >= start_utc) & (df_loaded["utc"] < stop_utc)
r_values = df_loaded.loc[mask, "R_km"]

fig,ax=plt.subplots(figsize=(8,6))
ax.hist(r_values, bins='auto')
ax.set_xlabel("R (km)")
ax.set_ylabel("Count")
ax.set_title("Distribution of R from 2015-09-01 to 2017-09-01")
plt.show()

## J2000

In [ ]:
cassini_location.attrs

In [ ]:
cassini_location

In [ ]:
## load in the location data of cassini from "cassini_j2000_saturn_centered_2004_2017_60s.pkl"
# change the working directory to the current file's directory

os.chdir(Path("/Users/chengzhimao/Documents/Code/saturn2025"))

with open("cassini_j2000_saturn_centered_2004_2017_60s.pkl", "rb") as f:
    cassini_location = pickle.load(f)

In [ ]:
cassini_location.head()

In [ ]:
type(pd.to_datetime('2010-12-21 00:50:00'))

## Trajectory Plot

### 2d

In [ ]:
# plot the location in x-y plane for given time range 
start_utc = pd.to_datetime("2010-12-21 00:50:00", utc=True)
end_utc  = pd.to_datetime("2017-12-22 02:00:00", utc=True)

#select the data in the given time range
mask = (cassini_location['utc'] >= start_utc) & (cassini_location['utc'] <= end_utc)
cassini_subset = cassini_location.loc[mask]

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(cassini_subset['x_Rs'], cassini_subset['y_Rs'], marker='o', linestyle='-', markersize=3)
ax.set_xlabel("x (R_S)")
ax.set_ylabel("y (R_S)")
ax.set_title("Cassini Location in x-y plane from 2010-12-21 to 2017-12-22")
plt.show()

In [ ]:
len(cassini_location)

In [ ]:
# plot the location in x-y plane for given time range with given point number
start_utc = pd.to_datetime("2010-12-21 00:50:00", utc=True)
end_utc  = pd.to_datetime("2017-12-22 02:00:00", utc=True)
n_points = 1000

#select the data in the given time range
mask = (cassini_location['utc'] >= start_utc) & (cassini_location['utc'] <= end_utc)
cassini_subset = cassini_location.loc[mask]

# select n_points evenly spaced points from cassini_subset
cassini_subset = cassini_subset.iloc[::len(cassini_subset) // n_points]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(cassini_subset['x_Rs'], cassini_subset['y_Rs'], marker='o', linestyle='-', markersize=3)
ax.set_xlabel("x (R_S)")
ax.set_ylabel("y (R_S)")
ax.set_title("Cassini Location in x-y plane from 2010-12-21 to 2017-12-22 (1000 points)")
plt.show()

### 3d

In [ ]:
# using matplotlib to plot the location in 3d space for given time range with given point number
from mpl_toolkits.mplot3d import Axes3D
start_utc = pd.to_datetime("2010-12-21 00:50:00", utc=True)
end_utc  = pd.to_datetime("2017-12-22 02:00:00", utc=True)
n_points = 1000 


#select the data in the given time range
mask = (cassini_location['utc'] >= start_utc) & (cassini_location['utc'] <= end_utc)
cassini_subset = cassini_location.loc[mask] 

# select n_points evenly spaced points from cassini_subset
cassini_subset = cassini_subset.iloc[::len(cassini_subset) // n_points]

In [ ]:
# using matplotlib 
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.plot(cassini_subset['x_Rs'], cassini_subset['y_Rs'], cassini_subset['z_Rs'], marker='o', linestyle='-', markersize=3)

ax.set_xlabel("x (R_S)")
ax.set_ylabel("y (R_S)")
ax.set_zlabel("z (R_S)")
ax.set_title("Cassini Location in 3D space from 2010-12-21 to 2017-12-22 (1000 points)")
plt.show()

In [ ]:
# using plotly to plot 3d location in space for given time range with given point number

In [ ]:

# using lines no markers
# colorbar means time from early to late

fig=go.Figure(data=[go.Scatter3d(
    x=cassini_subset['x_Rs'],
    y=cassini_subset['y_Rs'],
    z=cassini_subset['z_Rs'],
    mode='lines',
    line=dict(color='blue', width=2)
)]) 

fig.update_layout(
    title="Cassini Location in 3D space from 2010-12-21 to 2017-12-22 (1000 points)",
    scene=dict(
        xaxis_title='x (R_S)',
        yaxis_title='y (R_S)',
        zaxis_title='z (R_S)'
    )
)

In [ ]:
start_utc = pd.to_datetime("2004-12-21 00:50:00", utc=True)
end_utc  = pd.to_datetime("2017-12-22 02:00:00", utc=True)
n_points = 8000

# Select data in time range
mask = (cassini_location['utc'] >= start_utc) & (cassini_location['utc'] <= end_utc)
cassini_subset = cassini_location.loc[mask].copy()

# Select approximately n_points evenly spaced rows
cassini_subset = cassini_subset.iloc[::len(cassini_subset) // n_points]
# idx = np.linspace(0, len(cassini_subset) - 1, n_points).astype(int)
# cassini_subset = cassini_subset.iloc[idx].copy()

In [ ]:
# Convert UTC time to numeric values for coloring
time_numeric = cassini_subset['utc'].astype("int64") / 1e9  # seconds since epoch

fig = go.Figure(data=[go.Scatter3d(
    x=cassini_subset['x_Rs'],
    y=cassini_subset['y_Rs'],
    z=cassini_subset['z_Rs'],
    mode='lines',
    line=dict(
        color=time_numeric,
        colorscale='Viridis',
        width=3,
        colorbar=dict(
            title='UTC time',
            tickvals=[
                time_numeric.iloc[0],
                time_numeric.iloc[len(time_numeric)//2],
                time_numeric.iloc[-1]
            ],
            ticktext=[
                cassini_subset['utc'].iloc[0].strftime('%Y-%m-%d'),
                cassini_subset['utc'].iloc[len(cassini_subset)//2].strftime('%Y-%m-%d'),
                cassini_subset['utc'].iloc[-1].strftime('%Y-%m-%d')
            ]
        )
    )
)])

fig.update_layout(
    title=f"Cassini Location in 3D space from {start_utc.date()} to {end_utc.date()} ({n_points} points)",
    width=1200,
    height=800,
    scene=dict(
        xaxis_title='x (R_S)',
        yaxis_title='y (R_S)',
        zaxis_title='z (R_S)'
    ),
)

fig.show()

## KSM

In [ ]:
with open(Path("/Users/chengzhimao/Documents/Code/saturn2025/cassini_ksm_2004_2017_60s.pkl"), "rb") as f:
    cassini_location = pickle.load(f)

In [ ]:
df_all

In [ ]:
# Subsample for quick plotting
df_plot = df_all.iloc[::100]

theta = 2 * np.pi * (df_plot["MLT_hr"] / 24.0)

# Put 00 MLT at +x, 06 at +y, 12 at -x, 18 at -y for paper-like polar display
# Since theta=0 is 00 MLT:
x = df_plot["L_approx"] * np.cos(theta)
y = df_plot["L_approx"] * np.sin(theta)

plt.figure(figsize=(7, 7))
plt.plot(x, y)#, s=0.2)

# Enceladus and Dione approximate L-shell circles
for L, name in [(4.0, "Enceladus"), (6.2, "Dione")]:
    circle = plt.Circle((0, 0), L, fill=False, linestyle="--")
    plt.gca().add_patch(circle)
    plt.text(L, 0, name)

plt.axis("equal")
plt.xlim(-8, 8)
plt.ylim(-8, 8)

plt.xlabel("L, 00 MLT direction")
plt.ylabel("L, 06 MLT direction")
plt.title("Cassini coverage in approximate L-MLT coordinates")

plt.text(7.5, 0, "00", ha="left", va="center")
plt.text(0, 7.5, "06", ha="center", va="bottom")
plt.text(-7.5, 0, "12", ha="right", va="center")
plt.text(0, -7.5, "18", ha="center", va="top")

plt.show()

In [ ]:
start_utc = pd.to_datetime("2004-12-21 00:50:00", utc=True)
end_utc  = pd.to_datetime("2017-12-22 02:00:00", utc=True)
n_points = 8000

# Select data in time range
mask = (cassini_location['utc'] >= start_utc) & (cassini_location['utc'] <= end_utc)
cassini_subset = cassini_location.loc[mask].copy()

# Select approximately n_points evenly spaced rows
cassini_subset = cassini_subset.iloc[::len(cassini_subset) // n_points]

In [ ]:
cassini_location.head()

In [ ]:
# Convert UTC time to numeric values for coloring
time_numeric = cassini_subset['utc'].astype("int64") / 1e9  # seconds since epoch

fig = go.Figure(data=[go.Scatter3d(
    x=cassini_subset['x_ksm_Rs'],
    y=cassini_subset['y_ksm_Rs'],
    z=cassini_subset['z_ksm_Rs'],
    mode='lines',
    line=dict(
        color=time_numeric,
        colorscale='Viridis',
        width=3,
        colorbar=dict(
            title='UTC time',
            tickvals=[
                time_numeric.iloc[0],
                time_numeric.iloc[len(time_numeric)//2],
                time_numeric.iloc[-1]
            ],
            ticktext=[
                cassini_subset['utc'].iloc[0].strftime('%Y-%m-%d'),
                cassini_subset['utc'].iloc[len(cassini_subset)//2].strftime('%Y-%m-%d'),
                cassini_subset['utc'].iloc[-1].strftime('%Y-%m-%d')
            ]
        )
    )
)])

fig.update_layout(
    title=f"Cassini Location in 3D space from {start_utc.date()} to {end_utc.date()} ({n_points} points)",
    width=1200,
    height=800,
    scene=dict(
        xaxis_title='x (R_S)',
        yaxis_title='y (R_S)',
        zaxis_title='z (R_S)'
    ),
)

fig.show()

In [ ]:

# plot the radial distance of cassini distribution
fig,ax=plt.subplots(figsize=(12,6))
ax.hist(cassini_location['r_Rs'], bins='auto')
ax.set_xscale('log')
ax.set_xlabel('Radial Distance (Rs)')
ax.set_title('Distribution of Cassini Radial Distance from Saturn')
ax.set_ylabel('Count')